#### Read the data

In [13]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv("dataset/1_Daily_minimum_temps.csv")   # columns: Date, Temp

df["Temp"] = df["Temp"].astype(str).str.replace("?", "", regex=False)
df["Temp"] = df["Temp"].astype("float32")

temps = df["Temp"].values
temps

array([20.7, 17.9, 18.8, ..., 13.5, 15.7, 13. ],
      shape=(3650,), dtype=float32)

#### Normalize the data

In [14]:
scaler = MinMaxScaler()
temps_scaled = scaler.fit_transform(temps.reshape(-1, 1)).flatten()


#### Create sequences (12 days → next day)

In [15]:
import numpy as np

SEQ_LEN = 12
X_list, y_list = [], []

for i in range(len(temps_scaled) - SEQ_LEN):
    X_list.append(temps_scaled[i:i+SEQ_LEN])
    y_list.append(temps_scaled[i+SEQ_LEN])

X = np.array(X_list)
y = np.array(y_list)


#### Reshape X for RNN input

In [16]:
X = X.reshape(X.shape[0], X.shape[1], 1)


#### Split into train and validation sets

In [17]:
n = len(X)

train_end = int(n * 0.70)
val_end   = int(n * 0.85)

X_train = X[:train_end]
y_train = y[:train_end]

X_val = X[train_end:val_end]
y_val = y[train_end:val_end]

X_test = X[val_end:]
y_test = y[val_end:]


#### Build Encoder–Decoder RNN

In [18]:
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.SimpleRNN(64, return_sequences=False),
    tf.keras.layers.RepeatVector(1),
    tf.keras.layers.SimpleRNN(64, return_sequences=False),
    tf.keras.layers.Dense(1)
])


#### Compile the model

In [19]:
model.compile(optimizer="adam", loss="mse", metrics=["mae"])


#### Show model summary

In [20]:
model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ simple_rnn_2 (SimpleRNN)             │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ repeat_vector_1 (RepeatVector)       │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn_3 (SimpleRNN)             │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

#### Train the model

In [21]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32
)


Epoch 1/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - loss: 0.0148 - mae: 0.0922 - val_loss: 0.0089 - val_mae: 0.0738
Epoch 2/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0095 - mae: 0.0767 - val_loss: 0.0083 - val_mae: 0.0722
Epoch 3/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0094 - mae: 0.0765 - val_loss: 0.0090 - val_mae: 0.0735
Epoch 4/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0095 - mae: 0.0766 - val_loss: 0.0080 - val_mae: 0.0710
Epoch 5/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0093 - mae: 0.0754 - val_loss: 0.0082 - val_mae: 0.0710
Epoch 6/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0092 - mae: 0.0751 - val_loss: 0.0080 - val_mae: 0.0700
Epoch 7/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0093 - mae: 0.0759 - val_loss: 0.0082 - val_mae: 0.0708
Epoch 8/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0092 - mae: 0.0750 - val_loss: 0.0080 - val_mae: 0.0701
Epoch 9/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0093 -

#### Evaluate on the test set and train

In [22]:
train_loss, train_mae = model.evaluate(X_train, y_train, verbose=0)
test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)

print("Train MAE:", train_mae)
print("Test MAE:", test_mae)


Train MAE: 0.07606315612792969
Test MAE: 0.06974545121192932
